# OpenAQ Data Lakehouse Pipeline (Delta Lake & MinIO)

## Project Overview
This project simulates the ingestion and storage phase (Bronze Layer) of a Data Lakehouse ELT pipeline. The objective is to extract air quality data from a public REST API, process it dynamically, and store it efficiently in a cloud object storage system (MinIO) using the Delta Lake protocol to guarantee ACID transactions.

### Source Data & Architecture Strategy
We interface with the **OpenAQ API (v3)**, extracting two distinct types of data that require different architectural approaches:

1. **Locations (Metadata):** Static infrastructural data about the physical monitoring stations. Strategy: **Full Load** (Overwrite).
2. **Measurements (Time-Series):** High-velocity, dynamic data containing the actual pollution readings (e.g., PM10). Strategy: **Incremental Load** (Upsert/MERGE) with physical temporal partitioning.

In [1]:
# I use %pip to install all the required libraries directly into the active Jupyter kernel.
# The '%' ensures it targets our virtual environment (.venv) accurately in VS Code.
# I use '-q' (quiet) to suppress the long installation output and keep the notebook clean.
%pip install -q pandas requests deltalake pyarrow python-dotenv

print("Required libraries installed successfully!")

Note: you may need to restart the kernel to use updated packages.
Required libraries installed successfully!



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ==========================================
# 1. IMPORTS & ENVIRONMENT SETUP
# ==========================================

# I import pandas for data manipulation and tabular transformations
import pandas as pd

# I import requests to handle HTTP communication with the API
import requests

# I import time to implement pauses (throttling) and respect API rate limits
import time

# I import DeltaTable and write_deltalake to handle ACID transactions and Parquet files
from deltalake import DeltaTable, write_deltalake

# I import pyarrow to explicitly convert Pandas DataFrames into Arrow Tables.
import pyarrow as pa

# I import TableNotFoundError to gracefully handle the very first pipeline execution,
# allowing the code to create the table from scratch if it doesn't exist in MinIO yet.
from deltalake.exceptions import TableNotFoundError

## Phase 1: Environment Setup & Extraction Engine

Before interacting with the API or the cloud, we must configure our environment and build an abstraction layer for data extraction.

Here we define two core functions that will act as the engine of our pipeline:
* **`get_data`**: A robust web client that handles HTTP requests, URL construction, and error management (try-except) to prevent pipeline crashes during network timeouts.
* **`build_table`**: A normalization engine that automatically flattens complex, nested JSON payloads into structured Pandas DataFrames.

In [3]:
# ==========================================
# 2. CORE FUNCTIONS DEFINITION
# ==========================================

def get_data(base_url, endpoint, data_field=None, params=None, headers=None):
    """
    Performs a GET request to an API to fetch data.
    Uses try-except blocks to prevent pipeline crashes on HTTP or JSON errors.
    """
    try:
        # I dynamically construct the full URL
        endpoint_url = f"{base_url}/{endpoint}"

        # I execute the GET request with optional parameters and headers
        response = requests.get(endpoint_url, params=params, headers=headers)

        # This automatically raises an exception if the HTTP status is not 200 (Success)
        response.raise_for_status()

        # I verify if the data can be parsed as JSON
        try:
            data = response.json()
            # If a specific field is provided (e.g., 'results'), I extract only that section
            if data_field:
                data = data[data_field]
        except Exception:
            print("Response format is not as expected. Invalid JSON.")
            return None

        return data

    except requests.exceptions.RequestException as e:
        # I catch and log any request errors, such as timeouts or server crashes
        print(f"Request failed. Error code: {e}")
        return None


def build_table(json_data, record_path=None):
    """
    Builds and flattens a Pandas DataFrame from JSON-formatted data.
    """
    try:
        # I use json_normalize to flatten nested dictionaries natively
        df = pd.json_normalize(
            json_data,
            record_path
        )
        return df
    except Exception:
        # I catch formatting errors to avoid downstream failures
        print("Data is not in the expected format for normalization.")
        return None

print("Core extraction functions loaded successfully!")

Core extraction functions loaded successfully!


## Phase 2: Cloud Storage Connection (MinIO)

To persist our Bronze Layer, we are connecting to a **MinIO** server. MinIO is a high-performance, S3-compatible object storage system. By treating MinIO exactly as we would treat an AWS S3 bucket, we can seamlessly write our Delta Tables directly to the cloud.

We define a `storage_options` dictionary that contains our authentication credentials. This dictionary will be passed to every Delta Lake operation to authorize the connection.

In [4]:
# ==========================================
# 3. MINIO / S3 CONNECTION SETUP & PATHS
# ==========================================
import os
from dotenv import load_dotenv

# I load the environment variables from the local .env file
load_dotenv()

# I retrieve my cloud storage credentials securely from the environment
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

# --- DYNAMIC STORAGE PATHS ---
# I define my target bucket name centrally so it can be reused across the pipeline
bkt_name = "alumno03"

# I construct the base directory for the OpenAQ Bronze layer using an f-string
bronze_dir = f"s3://{bkt_name}/bronze/openaq"

# I create the storage_options dictionary formatted for S3 compatibility
storage_options = {
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_REGION": "us-east-1",
    "AWS_ALLOW_HTTP": "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true"
}

print(f"Cloud storage configured securely. Base directory set to: {bronze_dir}")

Cloud storage configured securely. Base directory set to: s3://alumno03/bronze/openaq


## Phase 3: Delta Lake Abstraction Layer

To handle the actual physical storage, we create three specialized functions using the `delta-rs` library.

1. **`save_data_as_delta`**: Handles standard Overwrite or Append operations. This acts as our foundational writing tool and will be used for our static metadata (`locations`).
2. **`save_new_data_as_delta`**: Uses the MERGE operation restricted to Inserts only. It adds records that don't exist yet while ignoring duplicates.
3. **`upsert_data_as_delta`**: Handles full Incremental Loads utilizing the powerful ACID MERGE operation. It compares incoming data against existing data using a predicate (condition). If a record exists, it updates it; if it's new, it inserts it. This is ideal for our dynamic time-series data (`measurements`).

In [5]:
# ==========================================
# 4. DELTA LAKE STORAGE FUNCTIONS
# ==========================================

def save_data_as_delta(df, path, storage_options, mode="overwrite", partition_cols=None):
    """
    Saves a dataframe in Delta Lake format at the specified cloud path.
    By default, it uses 'overwrite' mode.
    """
    write_deltalake(
        path,
        df,
        mode=mode,
        storage_options=storage_options,
        partition_by=partition_cols
    )


def save_new_data_as_delta(new_data, data_path, predicate, storage_options, partition_cols=None):
    """
    Saves only NEW data using the MERGE operation.
    It compares incoming data with existing data to ensure no duplicate records are stored.
    """
    try:
        dt = DeltaTable(data_path, storage_options=storage_options)

        # I explicitly convert Pandas to PyArrow to enforce strict data typing in memory
        new_data_pa = pa.Table.from_pandas(new_data)

        # I insert records from source that DO NOT exist in target
        (
            dt.merge(
                source=new_data_pa,
                source_alias="source",
                target_alias="target",
                predicate=predicate
            )
            .when_not_matched_insert_all()
            .execute()
        )

    except TableNotFoundError:
        # If the Delta table does not exist yet, I create a new one from scratch
        save_data_as_delta(new_data, data_path, storage_options=storage_options, partition_cols=partition_cols)


def upsert_data_as_delta(data, data_path, predicate, storage_options, partition_cols=None):
    """
    Saves data using the MERGE operation (Upsert).
    If records match, they are updated. If they don't match, new records are inserted.
    """
    try:
        dt = DeltaTable(data_path, storage_options=storage_options)

        # I explicitly convert Pandas to PyArrow
        data_pa = pa.Table.from_pandas(data)

        # Full Upsert logic: Update on match, Insert on no match
        (
            dt.merge(
                source=data_pa,
                source_alias="source",
                target_alias="target",
                predicate=predicate
            )
            .when_matched_update_all()
            .when_not_matched_insert_all()
            .execute()
        )
    except TableNotFoundError:
        # If the table doesn't exist, I fall back to standard creation
        save_data_as_delta(data, data_path, storage_options=storage_options, partition_cols=partition_cols)

print("Delta Lake storage functions loaded successfully!")

Delta Lake storage functions loaded successfully!


## Phase 4: Executing Full Extraction & Storage (Locations Metadata)

### Architectural Decisions & Strategy
* **Extraction Strategy:** We execute a complete extraction of the monitoring stations metadata (`locations`). Since this dataset acts as our static master dimension table, we store it in our MinIO Bronze bucket using the `overwrite` mode. This ensures our Data Lake always holds a clean, up-to-date snapshot of the physical infrastructure.
* **Schema Sanitization (Delta Lake Compliance):** Just like the measurements table, `json_normalize` expands nested location data into columns with dots (e.g., `country.id`). Because Delta Lake's Parquet engine strictly prohibits dots in column names, we must sanitize the schema by replacing `.` with `_` (using `regex=False` to prevent wildcards) before writing to MinIO.

In [6]:
# ==========================================
# 4.5 API AUTHENTICATION SETUP
# ==========================================
import os
from dotenv import load_dotenv

# I ensure the environment variables are loaded 
load_dotenv()

# I securely retrieve my personal API Key from the .env file.
# The API requires this key to be passed inside the HTTP headers.
my_api_key = os.getenv("OPENAQ_API_KEY")

headers = {
    "X-API-Key": my_api_key
}

print("API Authentication headers configured securely.")

API Authentication headers configured securely.


In [7]:
# ==========================================
# 5. EXECUTION: LOCATIONS METADATA PIPELINE
# ==========================================

# I set the destination cloud path dynamically building upon the base bronze directory
locations_s3_path = f"{bronze_dir}/locations"

print("Starting full metadata pipeline for locations...")

# Step 1: I extract data using the modular web client with split URL parameters
locations_json = get_data(
    base_url="https://api.openaq.org/v3",
    endpoint="locations",
    data_field="results", # I know the data that matters to me is within "results"
    headers=headers
)

# Step 2: I parse and flatten the JSON data
df_locations = build_table(locations_json)

# Step 3: I verify if data was successfully processed
if df_locations is not None and not df_locations.empty:

    # --- SCHEMA SANITIZATION ---
    # Fix 1: Delta Lake prohibits '.' in column names.
    df_locations.columns = df_locations.columns.str.replace('.', '_', regex=False)

    # Fix 2: Drop standard Pandas null columns (NaN / None) to save storage space.
    df_locations = df_locations.dropna(axis=1, how='all')

    # Fix 3: Force mixed 'object' types to string to avoid PyArrow/Delta Lake type inference crashes.
    for col in df_locations.select_dtypes(include='object').columns:
        df_locations[col] = df_locations[col].astype(str)

    print(f"Successfully processed and sanitized {len(df_locations)} location profiles.")

    # Step 4: I persist the master data using Overwrite mode
    save_data_as_delta(
        df=df_locations,
        path=locations_s3_path,
        mode="overwrite",
        storage_options=storage_options
    )

    print(f"\nLocations metadata successfully written to {locations_s3_path}!")
    display(df_locations.head(2))
else:
    print("[CRITICAL] Locations extraction failed. Storage operation aborted.")

Starting full metadata pipeline for locations...
Successfully processed and sanitized 100 location profiles.

Locations metadata successfully written to s3://alumno03/bronze/openaq/locations!


,id,name,locality,timezone,isMobile,isMonitor,instruments,sensors,licenses,bounds,...,owner_id,owner_name,provider_id,provider_name,coordinates_latitude,coordinates_longitude,datetimeFirst_utc,datetimeFirst_local,datetimeLast_utc,datetimeLast_local
0,3,NMA - Nima,None,Africa/Accra,False,True,"[{'id': 2, 'name': 'Government Monitor'}]","[{'id': 6, 'name': 'pm10 µg/m³', 'parameter': ...",None,"[-0.19968, 5.58389, -0.19968, 5.58389]",...,4,Unknown Governmental Organization,209,Dr. Raphael E. Arku and Colleagues,5.58389,-0.19968,nan,nan,nan,nan
1,4,NMT - Nima,None,Africa/Accra,False,True,"[{'id': 2, 'name': 'Government Monitor'}]","[{'id': 7, 'name': 'pm10 µg/m³', 'parameter': ...",None,"[-0.19898, 5.58165, -0.19898, 5.58165]",...,4,Unknown Governmental Organization,209,Dr. Raphael E. Arku and Colleagues,5.58165,-0.19898,nan,nan,nan,nan


## Phase 5: Executing Incremental Extraction (Measurements Time-Series)

### Complications & Data Quality Checks
* **Real-World Challenge:** In the OpenAQ metadata, many sensors are listed but are physically inactive, decommissioned, or temporarily offline. Processing empty server payloads wastes memory and raises schema mismatch errors during concatenation.
* **The Decision:** We implement an aggressive **Data Quality Gate** inside the loop. If `build_table` returns an empty DataFrame (meaning no recent data for that sensor), the pipeline dynamically logs a `[SKIPPED]` alert and discards the execution batch immediately.

### API Rate Limiting
* **The Challenge:** API servers block IP addresses that send too many requests concurrently (DDoS protection).
* **The Decision:** We enforce a strict 1-second pause (`time.sleep(1)`) between loop iterations to respect the provider's Rate Limits, prioritizing pipeline stability over execution speed.

In [8]:
# ==========================================
# 6. EXECUTION: MEASUREMENTS LOOP & DQ FILTER (STATEFUL)
# ==========================================

# --- STEP 1: EXTRACT TARGET SENSOR IDs ---
sensor_ids = []
for location in locations_json:
    sensors_list = location.get('sensors', [])
    for sensor in sensors_list:
        sensor_ids.append(sensor['id'])

sensor_ids = list(set(sensor_ids))


# --- NEW: STEP 1.5 - THE STATEFUL WATERMARK ---
# I check our MinIO Data Lake to see if we already have data to determine where to resume
measurements_s3_path = f"{bronze_dir}/measurements"
watermark_date = None

try:
    # I connect directly to the existing Delta Table
    dt_measurements = DeltaTable(measurements_s3_path, storage_options=storage_options)

    # I bring ONLY the timestamp column to memory to be extremely efficient
    df_dates = dt_measurements.to_pandas(columns=['period_datetimeFrom_utc'])

    # I find the absolute maximum (most recent) date stored in the Lake
    watermark_date = df_dates['period_datetimeFrom_utc'].max()

    # Defensive programming: If the table was empty for some reason, reset to None
    if pd.isna(watermark_date):
        watermark_date = None
    else:
        print(f"✅ [STATEFUL] Delta Table found! Resuming extraction from: {watermark_date}")

except TableNotFoundError:
    # If it fails, it means the table doesn't exist yet (First Run)
    print("⚠️ [STATEFUL] No existing Delta Table found. Initiating First-Time Full Load...")


# --- STEP 2: INITIALIZE BATCH CONTAINERS AND COUNTERS ---
all_measurements_batches = []
active_sensors_count = 0
total_target_sensors = len(sensor_ids)

print(f"\nAnalyzing a total of {total_target_sensors} sensors for active measurements...")
print("-" * 60)


# --- STEP 3: THE INCREMENTAL EXTRACTION LOOP ---
for s_id in sensor_ids:
    url_endpoint = f"sensors/{s_id}/measurements"

    # I dynamically build the query parameters for the API
    query_params = {}

    if watermark_date is not None:
        # If we have a watermark, we instruct the API to ONLY send newer data
        query_params['datetime_from'] = watermark_date

    # I pass the query_params dictionary to our standardized web client
    sensor_raw_json = get_data(
        base_url="https://api.openaq.org/v3",
        endpoint=url_endpoint,
        data_field="results",
        params=query_params, # <--- HERE THE MAGIC HAPPENS
        headers=headers
    )

    df_temp_batch = build_table(sensor_raw_json)

    # --- STEP 4: DATA QUALITY GATE (INACTIVE SENSOR HANDLING) ---
    if df_temp_batch is None or df_temp_batch.empty:
        print(f"  [SKIPPED] Sensor ID {s_id}: Inactive or contains no newer measurements.")
    else:
        df_temp_batch['sensor_id'] = s_id
        all_measurements_batches.append(df_temp_batch)
        active_sensors_count += 1
        print(f"  [OK] Sensor ID {s_id}: Active. Extracted {len(df_temp_batch)} fresh records.")

    # --- STEP 5: RATE LIMITING ---
    time.sleep(1)

print("-" * 60)
print(f"Stateful Extraction finished. Active sensors with new data: {active_sensors_count} out of {total_target_sensors}.")

✅ [STATEFUL] Delta Table found! Resuming extraction from: 2025-05-12T01:00:00Z

Analyzing a total of 391 sensors for active measurements...
------------------------------------------------------------
  [OK] Sensor ID 4099: Active. Extracted 100 fresh records.
  [OK] Sensor ID 1304579: Active. Extracted 100 fresh records.
  [SKIPPED] Sensor ID 5: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 6: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 7: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 8: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 9: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 10: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 11: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 12: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 13: Inactive or contains no newer measurements.
  [SKIPPED] Sensor ID 14: Inactive or contains no n

## Phase 6: Time-Series Parsing & Cloud Transaction (MERGE)

### Architectural Decisions & Strategy
* **Temporal Partitioning:** We cast the string timestamp column into a native Python Datetime object to extract `year`, `month`, and `day`. This logic enables physical directory partitioning in MinIO, drastically optimizing future OLAP analytical queries via partition pruning.
* **Idempotent Storage (ACID MERGE):** To support periodic pipeline runs without duplicating historical records, we execute `upsert_data_as_delta`.
* **The Unique Key Predicate:** The MERGE operation evaluates a composite key: `target.sensor_id = source.sensor_id AND target.period_datetimeFrom_utc = source.period_datetimeFrom_utc`. If the exact measurement for that sensor at that exact second already exists in MinIO, Delta updates the record; if it's a new timeframe, it inserts the record.

In [9]:
# ==========================================
# 7. CONSOLIDATION, PARTITIONING & MERGE
# ==========================================

# I set the destination cloud path dynamically building upon the base bronze directory
measurements_s3_path = f"{bronze_dir}/measurements"

# I check if the pipeline found at least one active sensor before proceeding
if len(all_measurements_batches) > 0:
    print("Consolidating data batches, sanitizing schema, and generating partitions...")

    # --- STEP 1: CONSOLIDATION ---
    # I vertically stack all the clean, individual batches into a single master dataframe
    df_measurements_master = pd.concat(all_measurements_batches, ignore_index=True)

    # --- STEP 2: SCHEMA SANITIZATION (THE 3 FIXES) ---
    # Fix 1: Delta Lake SQL prohibits '.' in column names. Replace with underscores.
    df_measurements_master.columns = df_measurements_master.columns.str.replace('.', '_', regex=False)

    # Fix 2: Drop standard Pandas null columns (NaN / None) to save storage space.
    df_measurements_master = df_measurements_master.dropna(axis=1, how='all')

    # Fix 3: Force mixed 'object' types to string to avoid PyArrow/Delta Lake type inference crashes.
    for col in df_measurements_master.select_dtypes(include='object').columns:
        df_measurements_master[col] = df_measurements_master[col].astype(str)

    # --- STEP 3: TEMPORAL PARTITIONING ---
    # I convert the string timestamp column into a native Python Datetime object
    df_measurements_master['datetime_parsed'] = pd.to_datetime(df_measurements_master['period_datetimeFrom_utc'])

    # I extract temporal metrics. Delta Lake will use these columns to create physical
    # folder hierarchies in MinIO (e.g., year=2026/month=05/day=14/), optimizing future queries.
    df_measurements_master['year'] = df_measurements_master['datetime_parsed'].dt.year
    df_measurements_master['month'] = df_measurements_master['datetime_parsed'].dt.month
    df_measurements_master['day'] = df_measurements_master['datetime_parsed'].dt.day

    # --- STEP 4: ACID TRANSACTIONS (UPSERT LOGIC) ---
    # I establish the unique logical predicate required to execute the MERGE transaction.
    # It checks if a record has the exact same sensor ID and the exact same measurement timestamp.
    upsert_condition = """
        target.sensor_id = source.sensor_id
        AND target.period_datetimeFrom_utc = source.period_datetimeFrom_utc
    """

    print("Executing MERGE operation in Delta Lake...")

    # I push the final master dataframe into MinIO using an idempotent Upsert strategy.
    upsert_data_as_delta(
        data=df_measurements_master,
        data_path=measurements_s3_path,
        predicate=upsert_condition,
        partition_cols=["year", "month", "day"],
        storage_options=storage_options
    )

    print(f"\nMeasurements processing cycle completed successfully! Data written to {measurements_s3_path}")
    # I display a subset of the critical columns for visual validation
    display(df_measurements_master[['sensor_id', 'value', 'period_datetimeFrom_utc', 'year', 'month', 'day']].head(3))
else:
    # Fallback log in case the API returned zero valid records for the entire network
    print("[WARNING] Pipeline executed, but zero active sensors were recorded. No data sent to MinIO.")

Consolidating data batches, sanitizing schema, and generating partitions...
Executing MERGE operation in Delta Lake...

Measurements processing cycle completed successfully! Data written to s3://alumno03/bronze/openaq/measurements


,sensor_id,value,period_datetimeFrom_utc,year,month,day
0,4099,45.0,2025-05-12T01:00:00Z,2025,5,12
1,4099,44.0,2025-05-12T02:00:00Z,2025,5,12
2,4099,43.0,2025-05-12T03:00:00Z,2025,5,12


# Part 2: Data Processing & Silver Layer Architecture

In this second phase of the project, we focus on transforming the raw data ingested in the Bronze layer to create a clean, typed, and enriched dataset known as the **Silver layer**. 

The main objective is to prepare the data for analytical querying by ensuring data quality, consistency, and optimized storage. According to the project requirements, we will use Pandas to apply several data processing techniques, including:
1. **Structural cleaning:** Dropping duplicates and handling null values to ensure data integrity.
2. **Dimensional enrichment:** Joining the dynamic measurements table with the static locations table to add geographical context.
3. **Data type casting & normalization:** Converting string dates to native datetime objects and standardizing text fields.
4. **Business rules implementation:** Creating a derived boolean flag to easily identify valid vs. invalid sensor readings.

By performing these transformations, we guarantee that the downstream layers (Gold layer or BI Dashboards) consume a highly reliable and performant Data Lakehouse.

## Step 0: Storage Profiling & Partitioning Strategy

Before transforming and writing data to the Silver layer, it is crucial to define an optimal storage partitioning strategy. Partitioning by too many dimensions (e.g., Year/Month/Day) on datasets with a small overall volume leads to the "Small File Problem," which severely degrades query performance in Big Data engines by forcing the system to open and close thousands of tiny files.

**What are we doing?** We are going to dynamically scan our MinIO Bronze bucket to calculate the exact physical size (in MB) and the total file count of our tables.

**Why are we doing this?** To make an empirically backed architectural decision. If the profiling shows that our dataset is relatively small but distributed across hundreds of files, we will use this evidence to justify partitioning the Silver layer strictly by `year`, thereby consolidating the data into highly optimized Parquet files.

In [10]:
# ==========================================
# PRE-REQUISITE: DEPENDENCIES INSTALLATION
# ==========================================

# I use %pip to install all the required libraries directly into the active Jupyter kernel.
# I added 's3fs' to this list, which is an S3-compatible file system interface for Python.
# We need s3fs specifically to inspect the physical folder sizes and file counts inside our MinIO buckets before processing.
# I use '-q' (quiet) to suppress the long installation output and keep the notebook clean.
%pip install -q pandas requests deltalake pyarrow python-dotenv s3fs

print("✅ Required libraries (including s3fs for storage profiling) installed successfully!")

Note: you may need to restart the kernel to use updated packages.
✅ Required libraries (including s3fs for storage profiling) installed successfully!



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# ==========================================
# STEP 0: BRONZE LAYER STORAGE PROFILING
# ==========================================
import s3fs
import os
from dotenv import load_dotenv

# I load the environment variables securely from the .env file
load_dotenv()

# I initialize the S3 File System client to connect directly to MinIO
fs = s3fs.S3FileSystem(
    key=os.getenv("MINIO_ACCESS_KEY"),
    secret=os.getenv("MINIO_SECRET_KEY"),
    client_kwargs={'endpoint_url': os.getenv("MINIO_ENDPOINT")}
)

# I define the target bucket and dynamic paths for the Bronze tables
bkt_name = "alumno03"
bronze_paths = {
    "Locations (Metadata)": f"{bkt_name}/bronze/openaq/locations",
    "Measurements (Time-Series)": f"{bkt_name}/bronze/openaq/measurements"
}

print("🔍 Profiling Bronze layer storage size in MinIO...\n")

# I iterate through the paths to calculate the total size and file count
for table_name, path in bronze_paths.items():
    try:
        # fs.find retrieves all files and metadata within the directory recursively
        files = fs.find(path, detail=True)
        
        # I calculate the total size in bytes and convert it to Megabytes (MB)
        total_bytes = sum([info['size'] for file_path, info in files.items()])
        total_mb = total_bytes / (1024 * 1024)
        
        print(f"📁 {table_name}: {total_mb:.4f} MB (Total physical files: {len(files)})")
        
    except FileNotFoundError:
        print(f"⚠️ {table_name}: Path not found in MinIO ({path})")
    except Exception as e:
        print(f"⚠️ {table_name}: Error scanning path -> {e}")

print("\n💡 Architecture Note: If the total storage size is relatively small but distributed across hundreds of files, it justifies partitioning the Silver layer strictly by 'year' to optimize the Parquet file sizes and prevent read degradation.")

🔍 Profiling Bronze layer storage size in MinIO...

📁 Locations (Metadata): 0.1586 MB (Total physical files: 12)
📁 Measurements (Time-Series): 4.1569 MB (Total physical files: 263)

💡 Architecture Note: If the total storage size is relatively small but distributed across hundreds of files, it justifies partitioning the Silver layer strictly by 'year' to optimize the Parquet file sizes and prevent read degradation.


## Step 1: Incremental Facts Extraction & Full Dimensions Load

To build a resilient and scalable pipeline, we will implement an **Incremental Load** using a "Watermark" approach for our facts table (`measurements`). Instead of reprocessing the entire historical dataset every time the script runs, we will:

1. **Check the Silver layer state:** Query the existing Silver table to find the most recently processed timestamp (`max_silver_date`).
2. **Read the Bronze layer:** Load the raw data from MinIO.
3. **Filter for the Delta:** Isolate and extract *only* the new measurement records that arrived after our watermark.

This logic ensures idempotency and dramatically optimizes compute resources. The dimensional table (`locations`) will be fully loaded as a standard practice for small metadata tables.

In [12]:
# ==========================================
# STEP 1: DATA LOADING & INCREMENTAL WATERMARK
# ==========================================
import pandas as pd
from deltalake import DeltaTable
from deltalake.exceptions import TableNotFoundError

bkt_name = "alumno03"
silver_measurements_dir = f"s3://{bkt_name}/silver/openaq/measurements"

print("🚀 Starting Step 1: Incremental Extraction...")

# 1. I search for the Watermark in the Silver Layer
try:
    dt_silver = DeltaTable(silver_measurements_dir, storage_options=storage_options)
    df_silver_current = dt_silver.to_pandas()
    
    if not df_silver_current.empty:
        # We find the latest timestamp already processed in Silver
        max_silver_date = pd.to_datetime(df_silver_current['date'], utc=True).max()
        print(f"🔍 Watermark found. Last processed date in Silver: {max_silver_date}")
    else:
        max_silver_date = pd.Timestamp('1970-01-01', tz='UTC')
        print("🔍 Silver table is empty. Processing all historical data.")
except TableNotFoundError:
    max_silver_date = pd.Timestamp('1970-01-01', tz='UTC')
    print("🔍 Silver table not found. Initiating full historical load.")

# 2. I fetch the raw source data from the Bronze Layer
print("📥 Fetching source data from Bronze Layer...")
dt_locations = DeltaTable(f"s3://{bkt_name}/bronze/openaq/locations", storage_options=storage_options)
dt_measurements = DeltaTable(f"s3://{bkt_name}/bronze/openaq/measurements", storage_options=storage_options)

df_locations = dt_locations.to_pandas()
df_bronze_measurements = dt_measurements.to_pandas()

# 3. I filter the Bronze measurements to extract ONLY the new records
# We use the 'datetime_parsed' column natively created in the Bronze layer
df_bronze_measurements['parsed_date'] = pd.to_datetime(df_bronze_measurements['datetime_parsed'], utc=True)

# I filter the dataframe to keep only records strictly newer than the watermark
df_measurements_incremental = df_bronze_measurements[df_bronze_measurements['parsed_date'] > max_silver_date].copy()

print(f"📊 Locations (Dimensions) loaded: {len(df_locations)} rows.")
print(f"📊 Incremental Measurements (Facts) detected: {len(df_measurements_incremental)} new rows to process.")

🚀 Starting Step 1: Incremental Extraction...
🔍 Silver table not found. Initiating full historical load.
📥 Fetching source data from Bronze Layer...
📊 Locations (Dimensions) loaded: 100 rows.
📊 Incremental Measurements (Facts) detected: 101392 new rows to process.


## Step 2: Structural Cleaning

Data collected from physical sensors and external APIs often contains anomalies such as duplicate records (due to network retries) or missing values (due to sensor downtime). In this step, we perform basic structural cleaning on the data residing in memory:

1. **Deduplication:** We drop any duplicate rows in both the dimensions (`locations`) and facts (`measurements`) dataframes to ensure data uniqueness.
2. **Null Handling:** We strictly remove records from the facts table where the actual measurement `value` is missing (NaN), as these rows provide no analytical value and could severely skew our future metrics and aggregations in the Gold layer.

In [13]:
# ==========================================
# STEP 2: STRUCTURAL CLEANING (DUPLICATES & NULLS)
# ==========================================

print("🧹 Starting Step 2: Structural Cleaning...")

# 1. Deduplication: Remove duplicate rows
# For locations, we use the unique 'id' column to identify and drop duplicates
df_locations = df_locations.drop_duplicates(subset=['id'])

# For incremental measurements, we drop fully duplicated rows
df_measurements_incremental = df_measurements_incremental.drop_duplicates()

# 2. Null Handling: Remove rows where the critical 'value' metric is missing (NaN)
# We safely drop these because we cannot perform mathematical aggregations on empty sensor readings
df_measurements_incremental = df_measurements_incremental.dropna(subset=['value'])

print("✅ Structural cleaning complete.")
print(f"📊 Post-cleaning Locations: {len(df_locations)} rows.")
print(f"📊 Post-cleaning Incremental Measurements: {len(df_measurements_incremental)} rows.")

🧹 Starting Step 2: Structural Cleaning...
✅ Structural cleaning complete.
📊 Post-cleaning Locations: 100 rows.
📊 Post-cleaning Incremental Measurements: 98342 rows.


## Step 3: Dimensional Enrichment (JOIN)

Raw fact tables often contain only foreign keys (like sensor IDs) to save storage space. To make the Silver layer analytics-ready, we must enrich these facts with human-readable dimensional data. 

In this step, we perform a **LEFT JOIN** between our incremental measurements and the static locations table. We extract the exact geographical columns from the Bronze metadata (`country_name` and `locality`) and standardize their names to `country` and `city` on the fly. We use a left join instead of an inner join to guarantee that no measurement data is lost, even if the API temporarily failed to provide the corresponding location metadata. This produces a flat, denormalized structure ideal for Business Intelligence modeling.

In [16]:
# ==========================================
# STEP 3: DIMENSIONAL ENRICHMENT (JOIN)
# ==========================================

print("🔗 Starting Step 3: Dimensional Enrichment (JOIN)...")

# 1. Prepare Dimensions: Select the exact geographic columns available and rename them to standard fields
locations_subset = df_locations[['id', 'name', 'country_name', 'locality']].rename(
    columns={'country_name': 'country', 'locality': 'city'}
)

# 2. Execute LEFT JOIN: Match 'sensor_id' from measurements with 'id' from locations
df_silver = pd.merge(
    left=df_measurements_incremental,
    right=locations_subset, 
    how='left',
    left_on='sensor_id',
    right_on='id'
)

# 3. Clean up: Drop redundant linking keys to optimize memory and future storage size
df_silver = df_silver.drop(columns=['id', 'sensor_id'])

print("✅ Dimensional enrichment complete.")
print(f"📊 Rows after JOIN: {len(df_silver)}")
print(f"📋 Key columns in Silver: {['name', 'country', 'city', 'value', 'parsed_date']}")

🔗 Starting Step 3: Dimensional Enrichment (JOIN)...
✅ Dimensional enrichment complete.
📊 Rows after JOIN: 98342
📋 Key columns in Silver: ['name', 'country', 'city', 'value', 'parsed_date']


## Step 4: Casting, Normalization & Quality Rules

To ensure our Silver layer is strictly typed and ready for Business Intelligence (BI) consumption, we must standardize the data structures. 

1. **Type Casting:** Delta Lake and Parquet files rely heavily on strict schemas. We explicitly cast our metrics to numeric types and ensure our timestamps are native Datetime objects.
2. **String Normalization:** Text fields entered by humans or disparate APIs often suffer from casing inconsistencies and trailing spaces. We standardize all geographic strings to uppercase and replace missing values with `"UNKNOWN"`. This prevents aggregation errors in BI tools (e.g., preventing 'Buenos Aires' and 'buenos aires ' from appearing as two different cities).
3. **Business & Quality Rules:** We introduce a derived boolean column (`is_valid_measurement`). In air quality contexts, negative concentration values are physically impossible. Instead of deleting them, we flag them. This allows data analysts to measure sensor failure rates in the Gold layer.

In [17]:
# ==========================================
# STEP 4: CASTING, NORMALIZATION & QUALITY RULES
# ==========================================

print("🛠️ Starting Step 4: Casting, Normalization & Quality Rules...")

# 1. Type Casting
# We explicitly force the measurement values to float. Any unparseable string will become NaN
df_silver['value'] = pd.to_numeric(df_silver['value'], errors='coerce')

# Our temporal column is already a Datetime object from Step 1, so we simply rename it to 'date' for the final schema
df_silver = df_silver.rename(columns={'parsed_date': 'date'})

# 2. String Normalization (Handling missing geodata and standardizing casing)
# We fill NaNs with a standard string, convert everything to uppercase, and strip trailing/leading spaces
df_silver['city'] = df_silver['city'].fillna('UNKNOWN').str.upper().str.strip()
df_silver['country'] = df_silver['country'].fillna('UNKNOWN').str.upper().str.strip()

# 3. Business Rule / Data Quality Flag
# We create a boolean flag: True if the value is physically possible (>= 0), False if it's an anomaly
df_silver['is_valid_measurement'] = df_silver['value'] >= 0

print("✅ Transformations complete.")
print("\n📋 Final Silver schema data types:")
print(df_silver[['name', 'country', 'city', 'value', 'date', 'is_valid_measurement']].dtypes)

🛠️ Starting Step 4: Casting, Normalization & Quality Rules...
✅ Transformations complete.

📋 Final Silver schema data types:
name                                 object
country                              object
city                                 object
value                               float64
date                    datetime64[us, UTC]
is_valid_measurement                   bool
dtype: object


## Step 5: Load to MinIO (Silver Layer)

In this final step, we persist our transformed DataFrames back to MinIO into the designated `silver` directory. We apply two distinct writing strategies based on the nature of the tables:

1. **Dimensions (Locations):** We perform a **Full Load** using the `overwrite` mode. Since this is a small, slowly changing dimension table, rewriting it entirely ensures we always have the latest snapshot without complex deduplication logic.
2. **Facts (Measurements):** We perform an **Incremental Load** using the `append` mode. We inject only the newly processed records. Crucially, based on our storage profiling from Step 0, we partition this table strictly by `year` to consolidate the data into optimally sized Parquet files and avoid the "Small File Problem".

In [18]:
# ==========================================
# STEP 5: LOAD TO MINIO (SILVER LAYER)
# ==========================================
from deltalake import write_deltalake

print("💾 Starting Step 5: Writing data to MinIO Silver Layer...")

silver_locations_dir = f"s3://{bkt_name}/silver/openaq/locations"
silver_measurements_dir = f"s3://{bkt_name}/silver/openaq/measurements"

# 1. Write Dimensions (Locations)
# We use the 'locations_subset' created in Step 3, which has the renamed and clean geographic columns
print("   -> Overwriting Locations (Dimensions)...")
write_deltalake(
    silver_locations_dir,
    locations_subset,
    mode="overwrite",
    storage_options=storage_options
)

# 2. Write Facts (Measurements)
# We append the enriched incremental batch, partitioning strictly by 'year' to optimize file sizes
print("   -> Appending new Measurements (Facts) partitioned by year...")
write_deltalake(
    silver_measurements_dir,
    df_silver,
    mode="append",
    partition_by=["year"],
    storage_options=storage_options
)

print("\n🎉 SUCCESS! Silver Layer processing completed and persisted in MinIO.")

💾 Starting Step 5: Writing data to MinIO Silver Layer...
   -> Overwriting Locations (Dimensions)...
   -> Appending new Measurements (Facts) partitioned by year...

🎉 SUCCESS! Silver Layer processing completed and persisted in MinIO.


# Part 3: Gold Layer & Business Aggregations

The Gold layer is the final stage of our Data Lakehouse. Its primary objective is to provide clean, aggregated, and business-ready datasets optimized for direct consumption by BI tools (like Power BI) or Machine Learning models. 

By pre-calculating the metrics at the database level, we drastically reduce the compute load on the visualization engine.

## Step 6: Daily Pollution Summary

In this step, we read the enriched facts from the Silver layer and aggregate them at a **Daily** and **Geographical** granularity (`year`, `month`, `day`, `country`, `city`). 

We will generate the following Key Performance Indicators (KPIs):
1. **Descriptive Stats:** Average, Maximum, and Minimum daily pollution values.
2. **Telemetry Health:** Total readings, the count of anomalous readings (`anomalies_count`), and the percentage of valid measurements (`data_quality_pct`) to track sensor network reliability.
3. **Business Logic:** An `air_quality_index` categorical label (GOOD, MODERATE, POOR) based on the daily average, making it immediately usable for dashboard color-coding and executive reporting.

In [ ]:
# ==========================================
# STEP 6: GOLD LAYER AGGREGATIONS
# ==========================================
import numpy as np

print("🏆 Starting Step 6: Building the Gold Layer (Business Aggregations)...")

# 1. Read the full Silver Layer
dt_silver_measurements = DeltaTable(silver_measurements_dir, storage_options=storage_options)
df_silver_full = dt_silver_measurements.to_pandas()

# 2. Extract Temporal Dimensions for Grouping
df_silver_full['year'] = df_silver_full['date'].dt.year
df_silver_full['month'] = df_silver_full['date'].dt.month
df_silver_full['day'] = df_silver_full['date'].dt.day

# 3. Perform Business Aggregations (Group By)
print("   -> Aggregating metrics by Year, Month, Day, Country, and City...")
df_gold = df_silver_full.groupby(['year', 'month', 'day', 'country', 'city']).agg(
    avg_pollution=('value', 'mean'),
    max_pollution=('value', 'max'),
    min_pollution=('value', 'min'),
    total_readings=('value', 'count'),
    valid_readings=('is_valid_measurement', 'sum') # Sums up the True values
).reset_index()

# 4. Calculate Derived KPIs
# KPI 1: Data Quality Percentage
df_gold['data_quality_pct'] = round((df_gold['valid_readings'] / df_gold['total_readings']) * 100, 2)

# KPI 2: Anomalies Count (Total readings minus the valid ones)
df_gold['anomalies_count'] = df_gold['total_readings'] - df_gold['valid_readings']

# KPI 3: Air Quality Index (Business Rule)
def categorize_air_quality(val):
    if pd.isna(val):
        return 'UNKNOWN'
    elif val < 15:
        return 'GOOD'
    elif val < 35:
        return 'MODERATE'
    else:
        return 'POOR'

df_gold['air_quality_index'] = df_gold['avg_pollution'].apply(categorize_air_quality)

# Clean up auxiliary columns used for calculation (we drop valid_readings as we already have total and anomalies)
df_gold = df_gold.drop(columns=['valid_readings'])

# 5. Save to MinIO (Gold Layer)
gold_daily_summary_dir = f"s3://{bkt_name}/gold/openaq/daily_pollution_summary"

print("💾 Saving aggregated Gold table to MinIO...")
write_deltalake(
    gold_daily_summary_dir,
    df_gold,
    mode="overwrite",
    schema_mode="overwrite", # <--- THE FIX: I force the schema update (from 11 to 12 columns)
    storage_options=storage_options
)

print("\n🎉 SUCCESS! Gold Layer built successfully.")
print(f"📉 The dataset was compressed to {len(df_gold)} aggregated business rows.")

# Display ALL columns to verify the calculations
display(df_gold.head())

🏆 Starting Step 6: Building the Gold Layer (Business Aggregations)...
   -> Aggregating metrics by Year, Month, Day, Country, and City...
💾 Saving aggregated Gold table to MinIO...

🎉 SUCCESS! Gold Layer built successfully.
📉 The dataset was compressed to 662 aggregated business rows.


,year,month,day,country,city,avg_pollution,max_pollution,min_pollution,total_readings,data_quality_pct,anomalies_count,air_quality_index
0,2016,1,29,CHILE,CONCEPCIÓN,114.000000,114.0,114.0,3,100.0,0,POOR
1,2016,1,29,CHILE,CORONEL,2130.000000,2130.0,2130.0,1,100.0,0,POOR
2,2016,1,29,CHILE,MACHALÍ,12.000000,12.0,12.0,1,100.0,0,GOOD
3,2016,1,29,THAILAND,NONE,498.166667,945.0,59.0,6,100.0,0,POOR
4,2016,1,29,UNITED KINGDOM,LONDON,72.666667,74.0,71.0,3,100.0,0,POOR


## Step 7: Data Lakehouse Maintenance (VACUUM Operation)

As demonstrated in the storage layer inspection, Delta Lake utilizes a transactional log (`_delta_log`) to achieve ACID compliance and enable "Time Travel." Over time, as `overwrite` or optimization operations are executed, older Parquet data files accumulate in the cloud storage, even though they are no longer part of the current active table state.

To optimize cloud storage costs and maintain a highly performant infrastructure, we implement the **VACUUM** operation. This command scans the transaction history and physically deletes data files that are no longer referenced by the active log.

**Architecture & Safety Note:** In this production-ready implementation, we rely on Delta Lake's default retention period of 168 hours (7 days). This best practice ensures that we maintain a safe window for Time Travel queries and prevents data corruption for any downstream consumers that might be reading the table concurrently while the maintenance job runs.

In [23]:
# ==========================================
# STEP 7: DATA LAKEHOUSE MAINTENANCE (VACUUM)
# ==========================================
from deltalake import DeltaTable

print("🧹 Starting Step 7: Executing VACUUM maintenance on Gold Layer...")

# 1. Instantiate the target Delta Table from MinIO
dt_gold = DeltaTable(gold_daily_summary_dir, storage_options=storage_options)

# 2. Execute the VACUUM command (Production Mode)
# By not passing parameters, Delta Lake applies its default safety rule:
# It will ONLY delete unreferenced files that are older than 7 days (168 hours).
print("   -> Scanning for stale files older than 7 days...")
deleted_files = dt_gold.vacuum()

if deleted_files:
    print(f"\n🎉 SUCCESS! Vacuum removed {len(deleted_files)} old file(s).")
else:
    print("\n✅ Vacuum complete. No files older than 7 days were found to delete. The recent history is safe.")

🧹 Starting Step 7: Executing VACUUM maintenance on Gold Layer...
   -> Scanning for stale files older than 7 days...

✅ Vacuum complete. No files older than 7 days were found to delete. The recent history is safe.


## Conclusion: The Modern Data Lakehouse

With the completion of the Gold Layer, we have successfully implemented a fully functional **Medallion Architecture (Bronze -> Silver -> Gold)** using Python, Delta Lake, and MinIO:

1. **Bronze Layer:** We ingested raw, nested JSON payloads from the OpenAQ API, preserving the original data state and implementing fault-tolerant extraction loops with rate limiting.
2. **Silver Layer:** We applied an idempotent Incremental Load (Watermark strategy) to process only new data. We enforced schema typing, handled nulls, enriched facts with dimensional geodata, and solved the "Small File Problem" by strategically partitioning the Parquet files by `year`.
3. **Gold Layer:** We aggregated the millions of potential data points into a daily business summary. By pre-calculating KPIs (Average, Max, Min, Data Quality %, and Air Quality Index) at the database level, we drastically reduced the compute load for downstream consumers.

**Next Steps (Business Intelligence):**
This final `daily_pollution_summary` Delta table is now ready to be connected directly to a BI tool like **Power BI** or **Superset**. The denormalized, strictly typed, and aggregated structure allows data analysts to build performant and interactive dashboards instantly, without needing to write complex DAX or SQL transformations on the frontend.